# 04. EuroCrops Dutch carrot development parcels

[Notebook 01](01_external_reference_audit.ipynb) introduces the external sources and their roles. EuroCrops adds a second carrot source for development, using Dutch 2020 production declarations for the validation top-up.

Saved outputs may describe an earlier run. [Notebook 08](08_dataset_summary.ipynb) shows the parcels used in the saved experiment.

## 1. Settings for this run

Review the settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `run_satellite_collection` | Enables Earth Engine submission and retrieval. |
| `submit_only` | Queues cloud work for retrieval in a later run. |
| `extra_batches` | Highest numbered extra slot to consider in the current round, after the regular batch. |
| `parcels_per_batch` | Candidate parcel limit for each new batch. |
| `parcel_file` | Repository-relative cache path for the published Dutch 2020 archive. |

Run All queues work and checks existing jobs on subsequent runs. The notebook plans one validation batch and checks original parcel footprints against saved development and test data. Preprocessing confirms matched pixels after imagery is available. [Collection guide](README.md)

In [9]:
# Edit these values before running collection.
run_satellite_collection = True
submit_only = False  # Queue missing exports and collect finished results on reruns.
extra_batches = 0  # One separately named top-up batch.
top_up_number = 2  # Resume the second top-up batch.
parcels_per_batch = 100
parcel_file = "data/external/eurocrops_zenodo/NL_2020.zip"


In [10]:
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup

parcel_path = root / parcel_file
notebook_setup.prepare_environment(root, install_missing=run_satellite_collection)
os.chdir(root)
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
report_root = (root / Path(settings.get("report_root", "data/pipeline_work/inventory")).expanduser()).resolve()
from crop_pipeline.notebook_display import show_table, show_mapping

Installing the cloned pipeline for fresh notebook kernels: .
Obtaining .
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for crop-pipeline (pyproject.toml): started
  Building editable for crop-pipeline (pyproject.toml): finished with status 'done'
  Created wheel for crop-pipeline: filename=crop_pipeline-0.1.0-0.editable-py3-none-any.whl size=1361 sha256=d5aa3caeced4d497ee2d5aff2d135852d22ff579fa7f4ceec63263ec234e38ad
  Stored in directory: /private/var/folders/wy/nfwt2j3n1v11v6kmszqs69cr0000gn/T/pip-ephem-wheel-cache-8fwbu01


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


## 2. Obtain references

Open the local annual GeoParquet. It contains parcel IDs, original crop codes and georeferenced polygon boundaries.

In [11]:
from importlib import reload

from crop_pipeline.sources import adaptive, annual, eurocrops
from crop_pipeline.windows.seasons import file_sha256

# Refresh the collection code if this notebook kernel was already open.
reload(eurocrops)
reload(adaptive)

if not parcel_path.is_file():
    eurocrops.cache_top_up_archive(parcel_path)
print("Local EuroCrops parcel file:", parcel_path)


Local EuroCrops parcel file: ./data/external/eurocrops_zenodo/NL_2020.zip


## 3. Check parcel eligibility

Keep valid Dutch polygons with carrot production codes 2717, 2783 or 2785. The resulting footprints supply the locations for Sentinel-2 collection.

In [12]:
features = eurocrops.load_carrots(parcel_path)
show_mapping({"Eligible carrot parcels": len(features)}, title="EuroCrops reference pool")


Item,Value
Eligible carrot parcels,"2,059"


## 4. Plan or resume batches

The planner uses eligible field footprints to create imagery requests. Saved plans carry their original selections and project splits. The batch table shows selected parcels and planned tiles for plans returned in this run.

In [13]:
batches = adaptive.prepare_round_batches(
    eurocrops.SOURCE, features, annual.archive_root(settings, report_root, repository_root=root), report_root,
    provenance={"dataset": "EuroCrops", "country": "NL", "year": eurocrops.TOP_UP_YEAR,
                "release_url": eurocrops.TOP_UP_URL, "parcel_file_sha256": file_sha256(parcel_path),
                "carrot_production_codes": sorted(eurocrops.CARROT_PRODUCTION_CODES)},
    parcels_per_crop=parcels_per_batch, candidate_crops=("carrot",), extra_batches=extra_batches,
    top_up_number=top_up_number,
    development_top_up={"carrot": ("train", "validation")},
)

# Count parcels and imagery tiles separately.
show_table([
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
      "Planned tiles": plan["estimate"]["tiles"]}
    for plan, context in batches if plan is not None
], title="Parcel and imagery workload by batch")

Development top-up: 0 candidate groups overlap saved test footprints.
EuroCrops top-up: 42 candidate groups overlap saved development footprints.
Round 2: resuming saved eurocrops selections and original collection plans.
Round 2: up to 100 candidates per crop in each new batch. Resume/submit 1 batches.


Batch,Selected parcels,Planned tiles
eurocrops_matched_top_up_carrot_validation_refs_1d65afbb,100,135


## 5. Collect annual imagery

Earth Engine uses the planned field footprints and years to find and process Sentinel-2 imagery. With `submit_only = True`, it can continue processing after submission finishes and the laptop is closed. On a later run, `submit_only = False` retrieves completed results, caches selected-pixel tables as Parquet and assembles annual TIFFs. Matching verified work can be reused. [Transport and caching](../../docs/reference/transport.md)

In [14]:
from crop_pipeline import source_workflow

project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


eurocrops 2020: selecting tiles 1–135 of 135
Scene inventories: 25/135 (0.0s)
Scene inventories: 50/135 (0.0s)
Scene inventories: 75/135 (0.1s)
Scene inventories: 100/135 (0.1s)
Scene inventories: 125/135 (0.1s)
Scene inventories: 135/135 (0.1s)
  Fetching page 1 (up to 5,000 rows; 0 saved so far)…
  Page 1 verified: 1,000 rows; 1,000 total (1.4s).
  Fetching page 2 (up to 5,000 rows; 1,000 saved so far)…
  Page 2 verified: 1,000 rows; 2,000 total (1.3s).
  Fetching page 3 (up to 5,000 rows; 2,000 saved so far)…
  Page 3 verified: 1,000 rows; 3,000 total (0.4s).
  Fetching page 4 (up to 5,000 rows; 3,000 saved so far)…
  Page 4 verified: 1,000 rows; 4,000 total (0.5s).
  Fetching page 5 (up to 5,000 rows; 4,000 saved so far)…
  Page 5 verified: 1,000 rows; 5,000 total (0.5s).
  Fetching page 6 (up to 5,000 rows; 5,000 saved so far)…
  Page 6 verified: 980 rows; 5,980 total (1.4s).
Verified and saved 5,980 result rows.
  Fetching page 1 (up to 5,000 rows; 0 saved so far)…
  Page 1 verif

## 6. Review collection status

Use the batch status to see which work finished and which can resume later. Verify the parcels and windows that pass imagery checks through preprocessing.

In [15]:
# Status comes from this run's collection response.
show_mapping({"Collection status": result["status"].replace("_", " ").title()},
             title="Collection result")
show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in result.get("batches", [])
], title="Collection status by batch")
source_workflow.print_summary(eurocrops.SOURCE, batches, result, "reuse", 10)

Item,Value
Collection status,Completed


Batch,Status
eurocrops_matched_top_up_carrot_validation_refs_1d65afbb,Completed


eurocrops — completed
References: reuse | Inward buffer: 10 m
Selected parcels: carrot 100
Batches: eurocrops_matched_top_up_carrot_validation_refs_1d65afbb: completed
Verified local tiles: 135/135 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: ./data/pipeline_work/adaptive_training/829e0cec79b8
Annual TIFF root: ./data/pipeline_work/annual_tifs
Saved registries: ./data/pipeline_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_matched_top_up_carrot_validation_refs_1d65afbb/source_handoffs/full_year_all_parcels/eurocrops.json
Next: Run Notebook 05 in preprocessing once the completed development and JECAM batches are available locally. Use data_collection/06 first for a saved dataset summary.


## Saved outputs

Browse reference files, batch states and imagery through the organized collection index.

In [16]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Organized collection outputs: ./pipeline_outputs/collection
Next: data_collection/08_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.


Continue to [Notebook 08](08_dataset_summary.ipynb) for the combined dataset profile and [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) for imagery quality and coverage.